# Audio Localization Dataset — Analysis
Analyzes all 72 WAV files across 8 angles, 4 mics, 2 durations.

In [ ]:
import os
import wave
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import librosa
import librosa.display
from scipy.signal import correlate
import warnings
warnings.filterwarnings('ignore')

BASE = os.path.dirname(os.path.abspath('__file__'))
ANGLES   = ['0', '45', '90', '135', '180', '225', '270', '315']
MICS     = ['mic_front', 'mic_back', 'mic_left', 'mic_right']
DURATIONS = ['3min', '5min']
SR = 16000

def load_wav(angle, source, duration, mic):
    path = os.path.join(BASE, angle, source, duration, mic + '.wav')
    with wave.open(path) as f:
        data = np.frombuffer(f.readframes(f.getnframes()), dtype=np.int16).astype(np.float32)
    return data / 32768.0

def rms_db(signal):
    return 20 * np.log10(np.sqrt(np.mean(signal**2)) + 1e-10)

def peak_db(signal):
    return 20 * np.log10(np.max(np.abs(signal)) + 1e-10)

def silence_pct(signal, threshold_db=-50, frame_size=1600):
    frames = signal[:len(signal) - len(signal) % frame_size].reshape(-1, frame_size)
    frame_rms = 20 * np.log10(np.sqrt(np.mean(frames**2, axis=1)) + 1e-10)
    return 100 * np.mean(frame_rms < threshold_db)

print('Setup complete.')

## 1. Dataset Overview — RMS, Peak, Silence per File

In [ ]:
import pandas as pd

records = []
for angle in ANGLES:
    for source in ['respeakerM', 'speakerM']:
        for dur in DURATIONS:
            for mic in MICS:
                path = os.path.join(BASE, angle, source, dur, mic + '.wav')
                if not os.path.exists(path):
                    continue
                sig = load_wav(angle, source, dur, mic)
                records.append({
                    'angle': int(angle), 'source': source, 'duration': dur, 'mic': mic,
                    'rms_db': round(rms_db(sig), 2),
                    'peak_db': round(peak_db(sig), 2),
                    'silence_pct': round(silence_pct(sig), 2),
                    'clip_pct': round(100 * np.mean(np.abs(sig) >= 1.0), 4)
                })

df = pd.DataFrame(records)
print(f'Total files analysed: {len(df)}')
df

## 2. RMS Level per Angle & Mic (5min recordings)

In [ ]:
df5 = df[(df['duration'] == '5min') & (df['source'] == 'speakerM')]

fig, ax = plt.subplots(figsize=(12, 5))
for mic in MICS:
    sub = df5[df5['mic'] == mic].sort_values('angle')
    ax.plot(sub['angle'], sub['rms_db'], marker='o', label=mic)

ax.set_xlabel('Angle (degrees)')
ax.set_ylabel('RMS (dBFS)')
ax.set_title('RMS Level per Angle — 5min recordings')
ax.set_xticks([int(a) for a in ANGLES])
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

## 3. Silence % per Angle (5min recordings)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for mic in MICS:
    sub = df5[df5['mic'] == mic].sort_values('angle')
    ax.plot(sub['angle'], sub['silence_pct'], marker='s', label=mic)

ax.set_xlabel('Angle (degrees)')
ax.set_ylabel('Silence (%)')
ax.set_title('Silence % per Angle — 5min recordings')
ax.set_xticks([int(a) for a in ANGLES])
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

## 4. Waveform + Spectrogram — All Angles, mic_front (5min)

In [ ]:
fig, axes = plt.subplots(len(ANGLES), 2, figsize=(16, 3 * len(ANGLES)))

for i, angle in enumerate(ANGLES):
    sig = load_wav(angle, 'speakerM', '5min', 'mic_front')
    t = np.linspace(0, len(sig) / SR, len(sig))

    # Waveform
    axes[i, 0].plot(t, sig, linewidth=0.3, color='steelblue')
    axes[i, 0].set_title(f'{angle}° — waveform (mic_front)')
    axes[i, 0].set_xlabel('Time (s)')
    axes[i, 0].set_ylabel('Amplitude')
    axes[i, 0].set_ylim(-1, 1)

    # Spectrogram
    D = librosa.amplitude_to_db(np.abs(librosa.stft(sig)), ref=np.max)
    librosa.display.specshow(D, sr=SR, x_axis='time', y_axis='hz', ax=axes[i, 1], cmap='magma')
    axes[i, 1].set_title(f'{angle}° — spectrogram (mic_front)')

plt.tight_layout()
plt.show()

## 5. All 4 Mics — Waveform Comparison at a Single Angle

In [ ]:
ANGLE = '0'   # change to any angle
DUR   = '5min'

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
for ax, mic in zip(axes, MICS):
    sig = load_wav(ANGLE, 'speakerM', DUR, mic)
    t = np.linspace(0, len(sig) / SR, len(sig))
    ax.plot(t, sig, linewidth=0.3)
    ax.set_title(mic)
    ax.set_ylabel('Amplitude')
    ax.set_ylim(-1, 1)
axes[-1].set_xlabel('Time (s)')
fig.suptitle(f'All Mics — {ANGLE}° {DUR}', fontsize=13)
plt.tight_layout()
plt.show()

## 6. GCC-PHAT — Time Delay of Arrival Between Mic Pairs

In [ ]:
def gcc_phat(sig1, sig2, sr=SR, max_delay_ms=2):
    """Generalized Cross-Correlation with Phase Transform."""
    n = len(sig1) + len(sig2) - 1
    S1 = np.fft.rfft(sig1, n=n)
    S2 = np.fft.rfft(sig2, n=n)
    R  = S1 * np.conj(S2)
    R /= (np.abs(R) + 1e-10)
    cc = np.fft.irfft(R, n=n)
    max_lag = int(sr * max_delay_ms / 1000)
    cc = np.concatenate([cc[-max_lag:], cc[:max_lag+1]])
    lags = np.arange(-max_lag, max_lag + 1) / sr * 1000  # ms
    return lags, cc

pairs = [('mic_front', 'mic_back'), ('mic_left', 'mic_right'), ('mic_front', 'mic_left')]

fig, axes = plt.subplots(len(ANGLES), len(pairs), figsize=(16, 3 * len(ANGLES)))

for i, angle in enumerate(ANGLES):
    for j, (m1, m2) in enumerate(pairs):
        s1 = load_wav(angle, 'speakerM', '5min', m1)
        s2 = load_wav(angle, 'speakerM', '5min', m2)
        lags, cc = gcc_phat(s1, s2)
        tdoa_ms = lags[np.argmax(cc)]
        axes[i, j].plot(lags, cc, linewidth=0.8)
        axes[i, j].axvline(tdoa_ms, color='red', linestyle='--', linewidth=1)
        axes[i, j].set_title(f'{angle}° | {m1} vs {m2} | TDOA={tdoa_ms:.2f}ms')
        axes[i, j].set_xlabel('Lag (ms)')

plt.tight_layout()
plt.show()

## 7. TDOA Summary Table — Estimated Direction per Angle

In [ ]:
print(f'{"Angle":>6} | {"Front-Back TDOA (ms)":>20} | {"Left-Right TDOA (ms)":>20}')
print('-' * 55)
for angle in ANGLES:
    s_front = load_wav(angle, 'speakerM', '5min', 'mic_front')
    s_back  = load_wav(angle, 'speakerM', '5min', 'mic_back')
    s_left  = load_wav(angle, 'speakerM', '5min', 'mic_left')
    s_right = load_wav(angle, 'speakerM', '5min', 'mic_right')

    lags_fb, cc_fb = gcc_phat(s_front, s_back)
    lags_lr, cc_lr = gcc_phat(s_left, s_right)

    tdoa_fb = lags_fb[np.argmax(cc_fb)]
    tdoa_lr = lags_lr[np.argmax(cc_lr)]

    print(f'{angle:>6}° | {tdoa_fb:>20.3f} | {tdoa_lr:>20.3f}')

## 8. 3min vs 5min Level Comparison per Angle

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, mic in zip(axes, ['mic_front', 'mic_right']):
    for dur, marker in [('3min', 'o'), ('5min', 's')]:
        vals = []
        for angle in ANGLES:
            path = os.path.join(BASE, angle, 'speakerM', dur, mic + '.wav')
            if os.path.exists(path):
                sig = load_wav(angle, 'speakerM', dur, mic)
                vals.append(rms_db(sig))
            else:
                vals.append(np.nan)
        ax.plot([int(a) for a in ANGLES], vals, marker=marker, label=dur)
    ax.set_title(f'RMS comparison — {mic}')
    ax.set_xlabel('Angle (degrees)')
    ax.set_ylabel('RMS (dBFS)')
    ax.set_xticks([int(a) for a in ANGLES])
    ax.legend()
    ax.grid(True)

plt.tight_layout()
plt.show()

## 9. Polar Plot — RMS Level vs Angle (mic_front, 5min)

In [ ]:
fig, ax = plt.subplots(subplot_kw={'projection': 'polar'}, figsize=(7, 7))

for mic in MICS:
    angles_rad = [np.deg2rad(int(a)) for a in ANGLES]
    rms_vals = []
    for angle in ANGLES:
        sig = load_wav(angle, 'speakerM', '5min', mic)
        rms_vals.append(rms_db(sig))
    # close the loop
    angles_rad.append(angles_rad[0])
    rms_vals.append(rms_vals[0])
    ax.plot(angles_rad, rms_vals, marker='o', label=mic)

ax.set_title('RMS (dBFS) vs Angle — 5min', va='bottom')
ax.set_theta_zero_location('N')
ax.set_theta_direction(-1)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()